# Занятие 10, пара 1. Перцептрон своими руками


Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai


**Как проходить тетрадь.** Основные примеры работают без ответов «Ваш ход».
        Если пропускаете упражнение, пропустите также его `check(...)`. Продолжайте следующий обычный раздел.
        Сохраняйте свои попытки в отдельной копии. Проверки помогают разобраться, но не выставляют баллы.
        Интернет, Git и регистрация для основной работы не нужны.


## Зачем нам обучаемый нейрон

В 09.2 мы сами придумывали признаки и считали центры классов. Теперь параметры решения будем
менять так, чтобы уменьшать ошибку. Начнём с одного нейрона и цифр 0/1, а во второй паре соберём сеть.
Python и циклы уже знакомы; подробно разберём именно смысл весов, потерь и обучения.
Нужны только NumPy и Matplotlib, данные лежат в комплекте. Другую тетрадь запускать не нужно.
Результат: один шаг обучения, численная проверка градиента, обученный классификатор и его ограничения.


### Подключим учебные файлы

        Следующая ячейка — готовая служебная настройка. Она ищет `course_support` в рабочей папке
        или выше. Её не надо переписывать по памяти. `DATA_DIR` указывает на учебные данные,
        `WORKSPACE_DIR` — на корень вашей копии, `check` запускает проверку упражнения.
        Если поддержка не найдена, перенесите весь комплект, а не только notebook.
        В этой тетради создаются все необходимые переменные: другую тетрадь запускать не нужно.


In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check, summary

PROJECT_DIR


## 1. Нейрон считает взвешенную сумму

Возьмём две измеренные характеристики изображения: заполненность левой и правой половины.
Они здесь условные, от 0 до 1. Вес показывает вклад признака в ответ: положительный подталкивает
к классу 1, отрицательный — к 0. Смещение $b$ сдвигает границу решения целиком.

$$z=x_1w_1+x_2w_2+b.$$

$z$ называют логитом: пока это любое число, не вероятность. Сначала явно перемножим и сложим.
При $x=(0.8,0.2)$, $w=(1,-1)$, $b=-0.1$ получим $z=0.5$.
Векторное `x @ weights` — ровно сумма попарных произведений, а не новая модель.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.array([.8, .2])
weights = np.array([1., -1.])
bias = -.1
contributions = x * weights
z = contributions.sum() + bias
print('Вклады:', contributions, 'z:', z)
print('То же через @:', x @ weights + bias)


Пороговый перцептрон выдаёт 1 при $z\geq0$, иначе 0. Историческое правило его обучения
работает с ошибкой класса; сегодня будем учить **гладкий логистический нейрон**, чтобы перейти
к градиентам. Это связанная, но не полностью одинаковая модель.

Сигмоида $\sigma(z)=1/(1+e^{-z})$ переводит логит в (0,1). Ответ трактуем как оценку вероятности
класса 1 внутри модели. Она не обязана быть хорошо откалибрована на реальном потоке.
При пороге 0.5 сигмоида и порог z=0 дают один класс, но плавный ответ позволяет вычислять производную.
`np.exp` — экспонента. Стабильную версию ниже пишем через `np.logaddexp(0,-z)`, которая считает
$\log(1+e^{-z})$ без переполнения при больших по модулю z.


In [ ]:
def sigmoid(z):
    return np.exp(-np.logaddexp(0, -np.asarray(z)))

print('Вероятность класса 1:', sigmoid(z))
line = np.linspace(-6, 6, 200)
fig, ax = plt.subplots(figsize=(9,4))
ax.plot(line, sigmoid(line), label='Сигмоида')
ax.step(line, (line >= 0).astype(float), where='post', label='Порог', alpha=.7)
ax.set(xlabel='Логит z', ylabel='Ответ', title='Граница та же, поведение вокруг неё разное')
ax.legend(); ax.grid(alpha=.3)
plt.show()


## 2. Что именно уменьшаем при обучении

Accuracy меняется скачком: вероятность 0.51 и 0.99 даёт один и тот же класс. Для обучения удобнее
плавная **функция потерь**. Для правильного класса 1 используем $-\log p$, для 0 — $-\log(1-p)$.
Это бинарная кросс-энтропия (BCE). Уверенная ошибка получает большой штраф.

Эквивалентная запись через логит: $L(z,y)=\log(1+e^z)-yz$.
`np.logaddexp(0,z)` вычисляет первый член устойчиво; не нужно брать логарифм округлённого нуля.
Это выбор критерия оптимизации, а не автоматическая гарантия качества.


In [ ]:
def binary_loss(logits, targets):
    losses = np.logaddexp(0, logits) - targets * logits
    return losses.mean()

for probability in [.51, .9, .01]:
    print('p(1) =', probability, 'loss при y=1 =', round(-np.log(probability), 3))
print('Loss нашего примера y=1:', binary_loss(np.array([z]), np.array([1.])))


## 3. Один шаг: почему вычитаем, а не прибавляем

Производная показывает, как loss меняется при небольшом увеличении параметра.
Для BCE с сигмоидой $\partial L/\partial z=p-y$. Через цепочку $z=xw+b$ получаем
$\partial L/\partial w_j=(p-y)x_j$, $\partial L/\partial b=p-y$.

В нашем примере y=1, p≈0.622: разность отрицательна. Чтобы уменьшить loss, надо увеличить z.
Вычитаем отрицательный градиент — вес положительного признака увеличивается.
Скорость обучения $\eta$ регулирует размер шага: $w_{new}=w-\eta\nabla L$.
Это не доля «выученного материала». Слишком большой шаг может проскочить хорошую область.


In [ ]:
target = 1.
probability = sigmoid(z)
error_signal = probability - target
grad_w = error_signal * x
grad_b = error_signal
learning_rate = .2
new_weights = weights - learning_rate * grad_w
new_bias = bias - learning_rate * grad_b
new_z = x @ new_weights + new_bias
print('Градиент весов:', grad_w, 'Градиент смещения:', grad_b)
print('Loss до:', binary_loss(z, target), 'После:', binary_loss(new_z, target))


### ✏️ Ваш ход — Чья правка слишком большая?

Для одного признака x=2, p=0.25, y=1 коллега получил градиент -1.5.
Он заменил единицы признака так, что x стал 20, но подобрал начальный вес в 10 раз меньше:
начальный логит и p сохранились. Вычислите `my_new_gradient`.
Какой `my_lr` нужен вместо исходного 0.1, чтобы **изменение логита от шага этого веса** осталось прежним?
Смещение не обновляем. Подсказка: изменение z равно x·изменение w; изменились оба множителя.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('10.1.1', my_new_gradient, my_lr)


**Мой вывод:** …


## 4. Проверим формулу независимо от себя

Можно немного увеличить вес, немного уменьшить и посмотреть, насколько изменился loss:
$\partial L/\partial w\approx[L(w+\varepsilon)-L(w-\varepsilon)]/(2\varepsilon)$.
Это численная проверка, а не удобный способ обучать сеть с миллионом весов.
Слишком маленький epsilon усиливает ошибки округления; слишком большой измеряет уже не локальный наклон.
Здесь берём 1e-5 и float64.


In [ ]:
epsilon = 1e-5
plus = weights.copy(); plus[0] += epsilon
minus = weights.copy(); minus[0] -= epsilon
loss_plus = binary_loss(x @ plus + bias, target)
loss_minus = binary_loss(x @ minus + bias, target)
numerical = (loss_plus - loss_minus) / (2 * epsilon)
print('По формуле:', grad_w[0], 'По двум изменениям:', numerical)
print('Близки:', np.isclose(grad_w[0], numerical))


### ✏️ Ваш ход — Найдите ошибку, а не просто замените знак

Чужой код использует `wrong = (y - sigmoid(x @ w)) * x`
как градиент, а затем выполняет `w -= lr * wrong`. Проверьте это на x=[1,-2], w=[.2,.1], y=1,
b=0: вычислите `my_correct_gradient` численно для каждого веса и `my_wrong_gradient` по чужой формуле.
Почему этот знак мог бы быть допустим в другом **правиле обновления**, но неверен здесь?
Для численной проверки используйте уже разобранную центральную разность.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('10.1.2', my_correct_gradient, my_wrong_gradient)


**Мой вывод:** …


## 5. Увидим спуск на карте

Сделаем маленькую симметричную задачу с двумя признаками, чтобы оба веса поместились на рисунке.
Для пачки примеров строки X — объекты, столбцы — признаки. `X @ w` возвращает логит каждой строки.
Средний градиент: $X^T(p-y)/N$. Транспонирование `.T` разворачивает матрицу, чтобы собрать
вклады каждого признака по всем объектам. Смещение в этой симметричной иллюстрации фиксировано нулём.

Сначала считаем путь, потом рисуем loss для сетки весов. `contourf` закрашивает области одинакового
уровня: это карта **параметров**, не изображение классов в пространстве исходных признаков.


In [ ]:
toy_x = np.array([[-1,-.5], [-.5,-1], [1,.5], [.5,1]], dtype=float)
toy_y = np.array([0,0,1,1], dtype=float)
toy_w = np.array([-1.5, 1.])
path = [toy_w.copy()]
for step in range(20):
    toy_p = sigmoid(toy_x @ toy_w)
    gradient = toy_x.T @ (toy_p - toy_y) / len(toy_y)
    toy_w = toy_w - .5 * gradient
    path.append(toy_w.copy())
path = np.array(path)
axis = np.linspace(-2, 3, 70)
surface = np.zeros((len(axis), len(axis)))
for row, w2 in enumerate(axis):
    for col, w1 in enumerate(axis):
        surface[row, col] = binary_loss(toy_x @ np.array([w1,w2]), toy_y)
fig, ax = plt.subplots(figsize=(7,6))
plot = ax.contourf(axis, axis, surface, levels=25, cmap='viridis')
ax.plot(path[:,0], path[:,1], 'o-', color='coral', markersize=4)
ax.scatter(path[0,0], path[0,1], color='white', s=90, label='Старт')
ax.set(xlabel='w1', ylabel='w2', title='Путь по поверхности средней BCE')
ax.legend(); fig.colorbar(plot, ax=ax, label='Loss')
plt.show()


У линейно разделимых данных без регуляризации минимум логистической потери может не достигаться
при конечных весах: увеличение уверенности продолжает уменьшать loss. Поэтому «веса всё ещё растут»
не обязательно ошибка цикла. В реальной работе нужны ограничения и проверка на отложенных данных.
Сейчас наша цель скромнее: увидеть связь формулы и движения на карте.


## 6. Теперь настоящие изображения, а не четыре точки

Берём тот же UCI Optdigits32, что в 09.2: исходные части train/validation сохраняем.
Выбираем только цифры 0 и 1. Это **более простая бинарная задача**: её accuracy нельзя напрямую
сравнить с распознаванием десяти классов в прошлой паре.

Картинки бинарные: 1 — штрих, 0 — фон. Превращаем 32×32 в 1024 признака `reshape(N,-1)`:
первое число сохраняет число картинок, -1 вычисляет длину строки. Порядок пикселей остаётся фиксированным.
Модель не знает соседство пикселей специально, она получает отдельный вес на каждую позицию.


In [ ]:
digits = np.load(DATA_DIR / 'digits' / 'optdigits32.npz', allow_pickle=False)
train_mask = np.isin(digits['train_labels'], [0,1])
val_mask = np.isin(digits['validation_labels'], [0,1])
train_images = digits['train_images'][train_mask]
val_images = digits['validation_images'][val_mask]
y_train = digits['train_labels'][train_mask].astype(float)
y_val = digits['validation_labels'][val_mask].astype(float)
x_train = train_images.reshape(len(train_images), -1).astype(float)
x_val = val_images.reshape(len(val_images), -1).astype(float)
print('Train:', x_train.shape, 'Validation:', x_val.shape)
fig, axes = plt.subplots(2,4,figsize=(11,6))
for ax, picture, label in zip(axes.ravel(), train_images[:8], y_train[:8]):
    ax.imshow(picture, cmap='gray', vmin=0, vmax=1, interpolation='nearest')
    ax.set_title('Цифра '+str(int(label))); ax.axis('off')
plt.tight_layout(); plt.show()


Собираем цикл из уже разобранного: логиты → loss → градиент → обновление.
Одна **эпоха** здесь — один проход по всей train-части. Мы используем полную пачку;
на следующей паре разделим её на мини-пачки. Validation только измеряем, по ней не считаем градиент.

Берём 80 эпох и lr=0.1 как фиксированный учебный опыт, а не подбираем их по test.
Сохраняем loss **после** обновления и для train, и для validation в одинаковом состоянии модели.
Простой ориентир — всегда предсказывать самый частый класс train.


In [ ]:
w = np.zeros(x_train.shape[1])
b = 0.
train_losses = []; val_losses = []
for epoch in range(80):
    logits = x_train @ w + b
    difference = sigmoid(logits) - y_train
    dw = x_train.T @ difference / len(y_train)
    db = difference.mean()
    w = w - .1 * dw
    b = b - .1 * db
    train_losses.append(binary_loss(x_train @ w + b, y_train))
    val_losses.append(binary_loss(x_val @ w + b, y_val))
val_probabilities = sigmoid(x_val @ w + b)
val_predictions = (val_probabilities >= .5).astype(int)
val_accuracy = (val_predictions == y_val).mean()
majority = int(y_train.mean() >= .5)
baseline_accuracy = (y_val == majority).mean()
print('Validation accuracy:', val_accuracy, 'Ориентир:', baseline_accuracy)
fig, axes = plt.subplots(1,2,figsize=(12,5))
axes[0].plot(range(1,81), train_losses, label='train')
axes[0].plot(range(1,81), val_losses, label='validation')
axes[0].set(xlabel='Эпоха', ylabel='BCE', title='Как менялась потеря'); axes[0].legend()
limit = np.abs(w).max()
plot = axes[1].imshow(w.reshape(32,32), cmap='coolwarm', vmin=-limit, vmax=limit)
axes[1].set_title('Вес каждой позиции: красный → 1, синий → 0')
fig.colorbar(plot, ax=axes[1]); plt.tight_layout(); plt.show()


Карта показывает вклад **добавления штриха в фиксированную позицию**, при прочих равных.
Это не тепловая карта внимания и не объяснение всех особенностей почерка. По форме можно обсудить,
почему середина и боковые части полезны для отличия 0 от 1. Попросите соседа предсказать знак веса
в одной области, затем сравните с рисунком.
Test не открываем ради настройки этой бинарной демонстрации. Для итогового сравнения классики,
MLP и будущей CNN нужна одна десятиклассовая задача и тот же test, с настройками, выбранными заранее.


### ✏️ Ваш ход — Дешёвая ошибка и дорогая ошибка

На **учебной validation** p(1)=[.2,.45,.55,.8], истинные классы
[0,1,0,1]. Ложная единица стоит 1 условную единицу, пропущенная единица — 5.
Сравните пороги .4 и .6: вычислите `my_costs` — словарь {порог: суммарная стоимость}.
Класс 1 выдаём при p>=порог. Выберите `my_threshold`. Почему одинаковая accuracy скрывает разницу?
В реальном проекте стоимость задаётся постановкой, а порог выбирается на validation, не test.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('10.1.3', my_costs, my_threshold)


**Мой вывод:** …


## Резерв и итог

Сдвиньте validation-изображения вправо на 2 пикселя **без циклического переноса**:
создайте zeros_like(val_images), присвойте `shifted[:,:,2:] = val_images[:,:,:-2]`.
Предскажите теми же весами, сравните доли ошибок и покажите 4 пары картинок крупно.
Если результат ухудшился, это свойство данного опыта, а не доказательство, что любой сдвиг всегда вреден.

Теперь мы умеем обучать один гладкий нейрон. Следующий шаг — скрытые нейроны и нелинейная активация:
они научатся строить признаки, а не только взвешивать исходные пиксели.
Данные и лицензия: `course_support/data/digits/SOURCE.md` (UCI, CC BY 4.0).
Для самостоятельного чтения: [Dive into Deep Learning, линейная классификация](https://d2l.ai/chapter_linear-classification/index.html).
На следующей паре производные будет считать autograd; понимание знака обновления всё равно пригодится.
